# Xử lý dữ liệu Xét nghiệm (Lab Results)

**Mục tiêu:** Trích xuất 10 đặc trưng xét nghiệm đại diện cho 24 giờ đầu tiên nhập ICU từ bảng `lab.csv` (dạng dọc — long format).

**Input:**
- `raw/eicu/lab.csv` — dữ liệu xét nghiệm gốc
- `processed/patient_clean.csv` — danh sách bệnh nhân hợp lệ

**Output:** `processed/lab_clean.csv`

**10 đặc trưng đầu ra:**
- `creatinine_max`
- `bun_mean`
- `lactate_mean`, `lactate_max`
- `wbc_mean`
- `hemoglobin_mean`
- `platelets_mean`
- `bilirubin_mean`
- `glucose_mean`, `glucose_max`

**Phương pháp:** Vì `lab.csv` ở dạng dọc (mỗi dòng = 1 xét nghiệm), ta chia thành 8 bảng nhỏ, tính toán riêng, rồi merge lại.

In [11]:
# ============================================================
# BƯỚC 1: KHỞI TẠO VÀ NẠP DỮ LIỆU
# ============================================================
# lab.csv ở dạng DỌC (long format): mỗi dòng là 1 xét nghiệm
# => Cần pivot/aggregate để có 1 dòng/bệnh nhân.
#
# Chỉ đọc 4 cột cần thiết để tiết kiệm RAM.

import pandas as pd
import numpy as np
import os

# Đường dẫn
LAB_PATH     = "../data/raw/eicu/lab.csv"
PATIENT_PATH = "../data/processed/patient_clean.csv"
OUTPUT_PATH  = "../data/processed/lab_clean.csv"

# Đọc danh sách bệnh nhân hợp lệ
df_patient = pd.read_csv(PATIENT_PATH)
valid_ids = set(df_patient["patientunitstayid"])
print(f"Số bệnh nhân hợp lệ: {len(valid_ids):,}")
del df_patient

# Đọc lab.csv (chỉ 4 cột cần thiết)
LAB_COLS = [
    "patientunitstayid",  # Mã định danh đợt nằm ICU (Khóa chính dùng để liên kết với bệnh nhân và các bảng khác)
    "labresultoffset",    # Mốc thời gian thực hiện xét nghiệm (Số phút tính từ thời điểm bệnh nhân nhập ICU = 0)
    "labname",            # Tên chỉ số xét nghiệm (Ví dụ: Glucose, BUN, Creatinine, WBC,...)
    "labresult",          # Giá trị kết quả xét nghiệm dưới dạng số (Ví dụ: nồng độ, số lượng tế bào đo được)
]
df_lab = pd.read_csv(LAB_PATH, usecols=LAB_COLS)

print(f"\nKích thước lab.csv gốc: {df_lab.shape}")
print("\n5 dòng đầu tiên:")
print(df_lab.info())
print(df_lab.head())

Số bệnh nhân hợp lệ: 2,326

Kích thước lab.csv gốc: (434660, 4)

5 dòng đầu tiên:
<class 'pandas.DataFrame'>
RangeIndex: 434660 entries, 0 to 434659
Data columns (total 4 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   patientunitstayid  434660 non-null  int64  
 1   labresultoffset    434660 non-null  int64  
 2   labname            434660 non-null  str    
 3   labresult          431939 non-null  float64
dtypes: float64(1), int64(2), str(1)
memory usage: 16.4 MB
None
   patientunitstayid  labresultoffset           labname  labresult
0            1754323             -647               Hct      38.30
1            1754323             -647  platelets x 1000     181.00
2            1754323             -647               RBC       4.86
3            1754323             -647            #NAME?       8.70
4            1754323             -647              MCHC      30.40


In [12]:
# ============================================================
# BƯỚC 2: LỌC CỬA SỔ 24H VÀ ÉP KIỂU SỐ
# ============================================================
# 1. Lọc thời gian: chỉ giữ xét nghiệm trong 24h đầu (0–1440 phút)
# 2. Ép labresult về dạng số (có thể có chuỗi lẫn vào)
# 3. Loại bỏ các dòng labresult = NaN sau ép kiểu

n_before = len(df_lab)

# Lọc 24h đầu
df_lab = df_lab[
    (df_lab["labresultoffset"] >= 0) &
    (df_lab["labresultoffset"] <= 1440)
].copy()
print(f"Sau lọc 24h: {len(df_lab):,} dòng "
      f"(giảm {n_before - len(df_lab):,})")

# Ép kiểu số
df_lab["labresult"] = pd.to_numeric(df_lab["labresult"], errors="coerce")
n_nan_after = df_lab["labresult"].isna().sum()
print(f"Số labresult = NaN sau ép kiểu: {n_nan_after:,}")

# Loại bỏ NaN
df_lab = df_lab.dropna(subset=["labresult"])
print(f"Sau khi loại NaN: {len(df_lab):,} dòng")

Sau lọc 24h: 98,870 dòng (giảm 335,790)
Số labresult = NaN sau ép kiểu: 702
Sau khi loại NaN: 98,168 dòng


In [13]:
# ============================================================
# BƯỚC 3: ÁNH XẠ TÊN XÉT NGHIỆM VỀ 8 NHÓM CHUẨN
# ============================================================
# labname trong eICU có nhiều biến thể (viết hoa, khoảng trắng,
# tên đồng nghĩa). Ta chuẩn hóa rồi map về 8 nhóm.

# Chuẩn hóa labname
df_lab["labname_clean"] = (
    df_lab["labname"]
    .str.lower()
    .str.strip()
)

# Từ điển mapping
LAB_MAP = {
    "creatinine":        "creatinine",
    "bun":               "bun",
    "lactate":           "lactate",
    "wbc":               "wbc",
    "wbc x 1000":        "wbc",
    "hemoglobin":        "hemoglobin",
    "hgb":               "hemoglobin",
    "platelets":         "platelets",
    "platelets x 1000":  "platelets",
    "bilirubin":         "bilirubin",
    "total bilirubin":   "bilirubin",
    "glucose":           "glucose",
    "bedside glucose":   "glucose",
}

df_lab["lab_clean"] = df_lab["labname_clean"].map(LAB_MAP)

# Chỉ giữ 8 nhóm chuẩn
df_lab = df_lab.dropna(subset=["lab_clean"]).copy()

print("Số bản ghi theo 8 nhóm xét nghiệm:")
print(df_lab["lab_clean"].value_counts())

print(f"\nTổng số dòng sau lọc: {len(df_lab):,}")
print(f"Số bệnh nhân có xét nghiệm: {df_lab['patientunitstayid'].nunique():,}")

Số bản ghi theo 8 nhóm xét nghiệm:
lab_clean
glucose       10065
creatinine     3209
bun            3203
hemoglobin     3123
wbc            2565
platelets      2552
bilirubin      1060
lactate         831
Name: count, dtype: int64

Tổng số dòng sau lọc: 26,608
Số bệnh nhân có xét nghiệm: 2,265


In [14]:
# ============================================================
# BƯỚC 4: TÍNH TOÁN RIÊNG TỪNG NHÓM XÉT NGHIỆM
# ============================================================
# Chia thành 8 DataFrame nhỏ, mỗi cái groupby theo patientunitstayid
# và tính mean/max tùy chỉ số.
#
# Đây là cách tường minh, dễ kiểm tra và debug hơn so với pivot_table.

# ---------- 1. Creatinine: max ----------
df_creatinine = (
    df_lab[df_lab["lab_clean"] == "creatinine"]
    .groupby("patientunitstayid")["labresult"]
    .max()
    .reset_index()
    .rename(columns={"labresult": "creatinine_max"})
)
print(f"Creatinine : {len(df_creatinine):,} bệnh nhân")

# ---------- 2. BUN: mean ----------
df_bun = (
    df_lab[df_lab["lab_clean"] == "bun"]
    .groupby("patientunitstayid")["labresult"]
    .mean()
    .reset_index()
    .rename(columns={"labresult": "bun_mean"})
)
print(f"BUN        : {len(df_bun):,} bệnh nhân")

# ---------- 3. Lactate: mean + max ----------
df_lactate = (
    df_lab[df_lab["lab_clean"] == "lactate"]
    .groupby("patientunitstayid")["labresult"]
    .agg(lactate_mean="mean", lactate_max="max")
    .reset_index()
)
print(f"Lactate    : {len(df_lactate):,} bệnh nhân")

# ---------- 4. WBC: mean ----------
df_wbc = (
    df_lab[df_lab["lab_clean"] == "wbc"]
    .groupby("patientunitstayid")["labresult"]
    .mean()
    .reset_index()
    .rename(columns={"labresult": "wbc_mean"})
)
print(f"WBC        : {len(df_wbc):,} bệnh nhân")

# ---------- 5. Hemoglobin: mean ----------
df_hemoglobin = (
    df_lab[df_lab["lab_clean"] == "hemoglobin"]
    .groupby("patientunitstayid")["labresult"]
    .mean()
    .reset_index()
    .rename(columns={"labresult": "hemoglobin_mean"})
)
print(f"Hemoglobin : {len(df_hemoglobin):,} bệnh nhân")

# ---------- 6. Platelets: mean ----------
df_platelets = (
    df_lab[df_lab["lab_clean"] == "platelets"]
    .groupby("patientunitstayid")["labresult"]
    .mean()
    .reset_index()
    .rename(columns={"labresult": "platelets_mean"})
)
print(f"Platelets  : {len(df_platelets):,} bệnh nhân")

# ---------- 7. Bilirubin: mean ----------
df_bilirubin = (
    df_lab[df_lab["lab_clean"] == "bilirubin"]
    .groupby("patientunitstayid")["labresult"]
    .mean()
    .reset_index()
    .rename(columns={"labresult": "bilirubin_mean"})
)
print(f"Bilirubin  : {len(df_bilirubin):,} bệnh nhân")

# ---------- 8. Glucose: mean + max ----------
df_glucose = (
    df_lab[df_lab["lab_clean"] == "glucose"]
    .groupby("patientunitstayid")["labresult"]
    .agg(glucose_mean="mean", glucose_max="max")
    .reset_index()
)
print(f"Glucose    : {len(df_glucose):,} bệnh nhân")

Creatinine : 2,092 bệnh nhân
BUN        : 2,089 bệnh nhân
Lactate    : 469 bệnh nhân
WBC        : 1,956 bệnh nhân
Hemoglobin : 1,970 bệnh nhân
Platelets  : 1,920 bệnh nhân
Bilirubin  : 938 bệnh nhân
Glucose    : 2,192 bệnh nhân


In [15]:
# ============================================================
# BƯỚC 5: HỢP NHẤT 8 BẢNG NHỎ
# ============================================================
# Merge 8 bảng nhỏ lại với nhau bằng how='outer' để giữ TẤT CẢ
# bệnh nhân có ít nhất 1 xét nghiệm.
# KHÔNG merge với patient_clean ở bước này.

df_lab_final = df_creatinine.copy()

for tbl in [df_bun, df_lactate, df_wbc, df_hemoglobin,
            df_platelets, df_bilirubin, df_glucose]:
    df_lab_final = df_lab_final.merge(
        tbl, on="patientunitstayid", how="outer"
    )

print(f"Kích thước sau merge 8 bảng: {df_lab_final.shape}")
print(f"Số bệnh nhân có ít nhất 1 xét nghiệm: {len(df_lab_final):,}")
print("\n5 dòng đầu tiên:")
display(df_lab_final.head())

print("\nThống kê mô tả 10 đặc trưng:")
print(df_lab_final.describe().round(2))

Kích thước sau merge 8 bảng: (2265, 11)
Số bệnh nhân có ít nhất 1 xét nghiệm: 2,265

5 dòng đầu tiên:


,patientunitstayid,creatinine_max,bun_mean,lactate_mean,lactate_max,wbc_mean,hemoglobin_mean,platelets_mean,bilirubin_mean,glucose_mean,glucose_max
0,141765,1.04,28.0,NaN,NaN,10.2,12.3,191.0,NaN,75.0,89.0
1,143870,1.14,13.5,NaN,NaN,11.7,12.3,152.0,NaN,131.5,140.0
2,144815,0.63,6.0,NaN,NaN,7.9,12.3,319.0,0.5,82.0,82.0
3,145427,1.05,14.0,NaN,NaN,21.1,13.3,196.0,NaN,128.5,139.0
4,147306,0.65,NaN,NaN,NaN,13.5,12.1,280.0,NaN,NaN,NaN



Thống kê mô tả 10 đặc trưng:
       patientunitstayid  creatinine_max  bun_mean  lactate_mean  lactate_max  \
count            2265.00         2092.00   2089.00        469.00       469.00   
mean          1704610.41            1.50     24.83          2.48         2.80   
std           1012481.78            1.58     18.61          2.76         3.14   
min            141765.00            0.10      2.00          0.30         0.30   
25%            845144.00            0.77     13.00          1.20         1.20   
50%           1631671.00            1.00     19.00          1.70         1.80   
75%           2714634.00            1.56     31.00          2.53         3.10   
max           3353113.00           20.20    197.60         27.90        27.90   

       wbc_mean  hemoglobin_mean  platelets_mean  bilirubin_mean  \
count   1956.00          1970.00         1920.00          938.00   
mean      11.63            11.15          203.42            1.24   
std       12.35             2.22    

In [16]:
# ============================================================
# BƯỚC 6: KIỂM TRA VÀ XUẤT FILE
# ============================================================

# ---------- Kiểm tra tính duy nhất ----------
n_dup = df_lab_final["patientunitstayid"].duplicated().sum()
print(f"Số patientunitstayid trùng lặp: {n_dup}")
assert n_dup == 0, "Có trùng lặp!"

# ---------- Tỷ lệ NaN ----------
print(f"\nTổng số bệnh nhân có xét nghiệm: {len(df_lab_final):,}")
print("\nTỷ lệ NaN của 10 đặc trưng:")
feature_cols = [c for c in df_lab_final.columns if c != "patientunitstayid"]
for col in feature_cols:
    n_miss = df_lab_final[col].isna().sum()
    pct = n_miss / len(df_lab_final) * 100
    print(f"  {col:20s}: {n_miss:>4,} ({pct:>5.2f}%)")

# ============================================================
# XỬ LÝ OUTLIER CỰC ĐOAN (bổ sung sau khi kiểm tra chất lượng)
# ============================================================
# WBC = 469.8 K/µL là gần như chắc chắn lỗi nhập liệu.
# Creatinine = 20.2 có thể là thật (suy thận nặng) -> giữ.

n_bad_wbc = (df_lab_final["wbc_mean"] > 100).sum()
print(f"\nWBC > 100 K/µL: {n_bad_wbc} dòng -> NaN")
df_lab_final.loc[df_lab_final["wbc_mean"] > 100, "wbc_mean"] = np.nan

# Kiểm tra lại sau sửa
print("Thống kê WBC sau làm sạch:")
print(df_lab_final["wbc_mean"].describe().round(2))

# ---------- Lưu file ----------
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df_lab_final.to_csv(OUTPUT_PATH, index=False)

print(f"\n✅ Đã lưu: {OUTPUT_PATH}")
print(f"   Kích thước cuối: {df_lab_final.shape}")

Số patientunitstayid trùng lặp: 0

Tổng số bệnh nhân có xét nghiệm: 2,265

Tỷ lệ NaN của 10 đặc trưng:
  creatinine_max      :  173 ( 7.64%)
  bun_mean            :  176 ( 7.77%)
  lactate_mean        : 1,796 (79.29%)
  lactate_max         : 1,796 (79.29%)
  wbc_mean            :  309 (13.64%)
  hemoglobin_mean     :  295 (13.02%)
  platelets_mean      :  345 (15.23%)
  bilirubin_mean      : 1,327 (58.59%)
  glucose_mean        :   73 ( 3.22%)
  glucose_max         :   73 ( 3.22%)

WBC > 100 K/µL: 1 dòng -> NaN
Thống kê WBC sau làm sạch:
count    1955.00
mean       11.40
std         6.71
min         0.10
25%         7.20
50%        10.00
75%        14.00
max        82.50
Name: wbc_mean, dtype: float64

✅ Đã lưu: ../data/processed/lab_clean.csv
   Kích thước cuối: (2265, 11)
